In [ ]:
import pandas as pd
from google.colab import files

# Upload all required CSV files
uploaded = files.upload()

Saving admission.csv to admission (2).csv
Saving bed.csv to bed (2).csv
Saving billing.csv to billing (2).csv
Saving department.csv to department (2).csv
Saving disease.csv to disease (2).csv
Saving doctor.csv to doctor (2).csv
Saving employee.csv to employee (2).csv
Saving patient.csv to patient (2).csv
Saving staff_assignment.csv to staff_assignment (2).csv
Saving ward.csv to ward (2).csv


In [ ]:
patients = pd.read_csv("patient.csv")
admissions = pd.read_csv("admission.csv")
departments = pd.read_csv("department.csv")
wards = pd.read_csv("ward.csv")
beds = pd.read_csv("bed.csv")
billing = pd.read_csv("billing.csv")
staff_assignment = pd.read_csv("staff_assignment.csv")
employee = pd.read_csv("employee.csv")
patient_insurance = pd.read_csv("insurance_provider.csv")
disease = pd.read_csv("disease.csv")
patient_diagnostic = pd.read_csv("diagnostic_test.csv")

In [ ]:
tables = {
    "Patients": patients,
    "Admissions": admissions,
    "Departments": departments,
    "Wards": wards,
    "Beds": beds,
    "Billing": billing,
    "Staff Assignment": staff_assignment,
    "Employee": employee,
    "Disease": disease,
}

for name, df in tables.items():
    print(name)
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

Patients
Rows: 30000
Columns: ['patient_id', 'gender', 'date_of_birth', 'blood_group', 'city', 'contact_number']
Admissions
Rows: 45000
Columns: ['admission_id', 'admission_date', 'discharge_date', 'admission_type', 'admission_status', 'patient_id', 'department_id', 'ward_id', 'bed_id', 'disease_id']
Departments
Rows: 11
Columns: ['department_id', 'department_name', 'department_type', 'floor_number', 'status']
Wards
Rows: 27
Columns: ['ward_id', 'ward_name', 'ward_type', 'total_beds', 'department_id']
Beds
Rows: 415
Columns: ['bed_id', 'bed_number', 'bed_status', 'ward_id']
Billing
Rows: 45000
Columns: ['bill_id', 'bill_date', 'total_amount', 'insurance_covered_amount', 'patient_payable_amount', 'payment_status', 'payment_mode', 'admission_id']
Staff Assignment
Rows: 207
Columns: ['assignment_id', 'employee_id', 'ward_id', 'shift']
Employee
Rows: 500
Columns: ['employee_id', 'employee_name', 'gender', 'role', 'employment_type', 'date_of_joining', 'department_id']
Disease
Rows: 20
Colum

In [ ]:

raw_master = admissions.copy()
raw_master = raw_master.merge(
    patients,
    on="patient_id",
    how="left"
)
raw_master = raw_master.merge(
    departments,
    on="department_id",
    how="left",
    suffixes=("", "_department")
)
raw_master = raw_master.merge(
    wards,
    on="ward_id",
    how="left",
    suffixes=("", "_ward")
)
raw_master = raw_master.merge(
    beds,
    on="bed_id",
    how="left",
    suffixes=("", "_bed")
)
raw_master = raw_master.merge(
    disease,
    on="disease_id",
    how="left",
    suffixes=("", "_disease")
)
billing_summary = (
    billing
    .groupby("admission_id")
    .agg(
        Total_Billing=("total_amount", "sum"),
        Insurance_Covered_Amount=(
            "insurance_covered_amount",
            "sum"
        ),
        Patient_Payable_Amount=(
            "patient_payable_amount",
            "sum"
        )
    )
    .reset_index()
)

raw_master = raw_master.merge(
    billing_summary,
    on="admission_id",
    how="left"
)

staff_summary = (
    staff_assignment
    .groupby("ward_id")
    .agg(
        Staff_Count=(
            "employee_id",
            "nunique"
        ),
        Staff_Assignment_Count=(
            "assignment_id",
            "nunique"
        )
    )
    .reset_index()
)

raw_master = raw_master.merge(
    staff_summary,
    on="ward_id",
    how="left"
)


employee_summary = (
    employee
    .groupby("department_id")
    .agg(
        Employee_Count=(
            "employee_id",
            "nunique"
        )
    )
    .reset_index()
)

raw_master = raw_master.merge(
    employee_summary,
    on="department_id",
    how="left"
)



In [ ]:

print("RAW DATA INTEGRATION COMPLETED")
print("Total Rows:", len(raw_master))
print(
    "Total Columns:",
    len(raw_master.columns)
)



RAW DATA INTEGRATION COMPLETED
Total Rows: 45000
Total Columns: 34


In [ ]:
file_name = "hospital_raw_integrated.xlsx"

raw_master.to_excel(
    file_name,
    index=False
)

print(
    "\n✅ File saved successfully:",
    file_name
)
files.download(file_name)


✅ File saved successfully: hospital_raw_integrated.xlsx


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Cleaning DataSet


In [38]:
import pandas as pd
import numpy as np
df = pd.read_excel("/hospital_dirty_dataset.xlsx")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
display(df.head())

Rows: 45900
Columns: 34


,admission_id,admission_date,discharge_date,admission_type,admission_status,patient_id,department_id,ward_id,bed_id,disease_id,...,bed_status,ward_id_bed,disease_name,disease_category,Total_Billing,Insurance_Covered_Amount,Patient_Payable_Amount,Staff_Count,Staff_Assignment_Count,Employee_Count
0,1,2020-02-25,2020-02-27,Emergency,Discharged,166,2,6,76,10,...,Available,6,Anemia,Hematological,68483,61634.7,6848.3,7,7,54
1,2,2022-02-22,2022-03-04,Elective,Discharged,8622,5,21,302,11,...,Available,21,Fracture Femur,Orthopedic,70917,35458.5,35458.5,7,7,40
2,3,2021-02-03,2021-02-09,Elective,Discharged,23976,1,2,11,9,...,Available,2,Chronic Obstructive Pulmonary Disease,Respiratory,28137,25323.3,2813.7,11,11,51
3,4,2021-12-31,2022-01-05,Elective,Discharged,16635,2,10,128,1,...,Available,10,Acute Myocardial Infarction,Cardiac,80665,64532.0,16133.0,6,6,54
4,5,2022-07-02,2022-07-07,Elective,Discharged,10654,3,11,157,7,...,Available,11,Hypertension,Cardiac,54920,27460.0,27460.0,5,5,43


In [39]:
# Check complete duplicate records
duplicate_count = df.duplicated().sum()
print("Duplicate records BEFORE cleaning:", duplicate_count)
if duplicate_count > 0:
    display(
        df[df.duplicated(keep=False)].head(20)
    )
else:
    print("No duplicate records found.")

Duplicate records BEFORE cleaning: 900


,admission_id,admission_date,discharge_date,admission_type,admission_status,patient_id,department_id,ward_id,bed_id,disease_id,...,bed_status,ward_id_bed,disease_name,disease_category,Total_Billing,Insurance_Covered_Amount,Patient_Payable_Amount,Staff_Count,Staff_Assignment_Count,Employee_Count
4,5,2022-07-02,2022-07-07,Elective,Discharged,10654,3,11,157,7,...,Available,11,Hypertension,Cardiac,54920,27460.0,27460.0,5,5,43
24,17920,2025-07-23,2025-07-27,Elective,Discharged,13377,1,4,51,18,...,Available,4,Viral Fever,Infectious,16352,8176.0,8176.0,5,5,51
96,25947,2024-07-27,2024-08-05,Elective,Discharged,20499,1,1,3,2,...,Available,1,Stroke,Neurological,13089,10471.2,2617.8,8,8,51
140,10407,2020-08-30,2020-09-05,Elective,Discharged,6627,1,3,32,10,...,Available,3,Anemia,Hematological,38413,34571.7,3841.3,7,7,51
176,174,2022-08-08,2022-08-12,Elective,Discharged,21718,3,13,191,3,...,Available,13,Road Traffic Accident,Trauma,15427,13884.3,1542.7,10,10,43
198,11643,2021-09-13,2021-09-19,Elective,Discharged,23193,3,12,180,19,...,Available,12,COVID-19,Infectious,48600,43740.0,4860.0,12,12,43
203,31453,2021-01-11,2021-01-15,Elective,Discharged,24668,4,20,286,17,...,Available,20,Urinary Tract Infection,Infectious,12681,10144.8,2536.2,6,6,33
221,217,2024-07-03,2024-07-08,Elective,Discharged,7175,3,11,157,2,...,Available,11,Stroke,Neurological,7467,5973.6,1493.4,5,5,43
236,232,2021-10-12,2021-10-14,Emergency,Discharged,7708,4,18,263,18,...,Available,18,Viral Fever,Infectious,33056,29750.4,3305.6,5,5,33
252,16881,2021-01-08,2021-01-16,Elective,Discharged,17022,5,21,308,20,...,Available,21,Tuberculosis,Infectious,42877,0.0,42877.0,7,7,40


In [40]:
if duplicate_count > 0:
    df = df.drop_duplicates().copy()
    print("Duplicate records removed.")
else:
    print("No duplicates to remove.")

Duplicate records removed.


In [41]:
remaining_duplicates = df.duplicated().sum()

print(
    "Duplicate records AFTER cleaning:",
    remaining_duplicates
)

if remaining_duplicates == 0:
    print(" Duplicate cleaning completed successfully.")

Duplicate records AFTER cleaning: 0
 Duplicate cleaning completed successfully.


In [42]:
#Column-wise missing values
missing_count = df.isnull().sum()

print(" MISSING VALUES BEFORE CLEANING")

display(
    missing_count[missing_count > 0]
)

 MISSING VALUES BEFORE CLEANING


,0
gender,450
date_of_birth,450
blood_group,450
city,450
contact_number,450


In [43]:
missing_percentage = (
    df.isnull().sum() / len(df)
) * 100

print(" MISSING PERCENTAGE BEFORE CLEANING")

display(
    missing_percentage[missing_percentage > 0]
)

 MISSING PERCENTAGE BEFORE CLEANING


,0
gender,1.0
date_of_birth,1.0
blood_group,1.0
city,1.0
contact_number,1.0


In [44]:
# Overall missing percentage
total_missing = df.isnull().sum().sum()
total_cells = df.shape[0] * df.shape[1]

overall_missing_percentage = (
    total_missing / total_cells
) * 100

print("Total Missing Values:", total_missing)
print("Total Cells:", total_cells)
print(
    "Overall Missing Percentage:",
    round(overall_missing_percentage, 2),
    "%"
)

Total Missing Values: 2250
Total Cells: 1530000
Overall Missing Percentage: 0.15 %


In [45]:
# Handling
numeric_columns = df.select_dtypes(
    include=np.number
).columns

text_columns = df.select_dtypes(
    include="object"
).columns
# Numeric columns → median
for col in numeric_columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(
            df[col].median()
        )
# Text columns → mode
for col in text_columns:
    if df[col].isnull().sum() > 0:
        mode_value = df[col].mode()

        if len(mode_value) > 0:
            df[col] = df[col].fillna(
                mode_value[0]
            )

In [46]:
remaining_missing = df.isnull().sum().sum()
print(
    "Total missing values AFTER cleaning:",
    remaining_missing
)

if remaining_missing == 0:
    print(" Missing-value cleaning completed successfully.")
else:
    print(" Missing values still remain.")

    display(
        df.isnull().sum()[
            df.isnull().sum() > 0
        ]
    )

Total missing values AFTER cleaning: 0
 Missing-value cleaning completed successfully.


In [47]:
# Check spaces BEFORE standardization
print("SPACE CHECK BEFORE CLEANING")
space_count = 0
for col in text_columns:
    count = (
        df[col].notna()
        & (df[col] != df[col].str.strip())
    ).sum()
    if count > 0:
        print(
            col,
            ":",
            count,
            "records with leading/trailing spaces"
        )

        space_count += count

print(
    "\nTotal records containing spaces:",
    space_count
)

SPACE CHECK BEFORE CLEANING
admission_type : 248 records with leading/trailing spaces
admission_status : 239 records with leading/trailing spaces
gender : 222 records with leading/trailing spaces
department_name : 567 records with leading/trailing spaces
department_type : 231 records with leading/trailing spaces
ward_type : 207 records with leading/trailing spaces
bed_status : 214 records with leading/trailing spaces
disease_category : 213 records with leading/trailing spaces

Total records containing spaces: 2141


In [48]:
# Removing
for col in text_columns:
    df[col] = df[col].str.strip()

In [49]:
remaining_spaces = 0
for col in text_columns:
    remaining_spaces += (
        df[col].notna()
        & (df[col] != df[col].str.strip())
    ).sum()

print(
    "Remaining leading/trailing spaces:",
    remaining_spaces
)

if remaining_spaces == 0:
    print("Space cleaning completed.")

Remaining leading/trailing spaces: 0
Space cleaning completed.


In [50]:
# Standardize Department Names
print("DEPARTMENT NAMES BEFORE STANDARDIZATION")

display(
    df["department_name"]
    .value_counts()
)

DEPARTMENT NAMES BEFORE STANDARDIZATION


,count
department_name,
Surgery,9759
Emergency,8449
Pediatrics,8148
Internal Medicine,7394
Orthopedics,5686
ICU,3943
SURGERY,132
surgery,122
emergency,114


In [51]:
df["department_name"] = (
    df["department_name"]
    .astype(str)
    .str.strip()
    .str.title()
)

In [52]:
print("DEPARTMENT NAMES AFTER STANDARDIZATION")
display(
    df["department_name"]
    .value_counts()
)

DEPARTMENT NAMES AFTER STANDARDIZATION


,count
department_name,
Surgery,10126
Emergency,8777
Pediatrics,8438
Internal Medicine,7695
Orthopedics,5924
Icu,4040


In [53]:
# Data Types
print("DATA TYPES BEFORE CORRECTION")

display(df.dtypes)

DATA TYPES BEFORE CORRECTION


,0
admission_id,int64
admission_date,object
discharge_date,object
admission_type,object
admission_status,object
patient_id,int64
department_id,int64
ward_id,int64
bed_id,int64
disease_id,int64


In [54]:
# changing data types
date_columns = [
    "admission_date",
    "discharge_date",
    "date_of_birth"
]

for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(
            df[col],
            errors="coerce"
        )
numeric_cols = [
    "floor_number",
    "total_beds",
    "Total_Billing",
    "Insurance_Covered_Amount",
    "Patient_Payable_Amount",
    "Staff_Count",
    "Staff_Assignment_Count",
    "Employee_Count"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

In [55]:
print("DATA TYPES AFTER CORRECTION")
display(df.dtypes)

DATA TYPES AFTER CORRECTION


,0
admission_id,int64
admission_date,datetime64[ns]
discharge_date,datetime64[ns]
admission_type,object
admission_status,object
patient_id,int64
department_id,int64
ward_id,int64
bed_id,int64
disease_id,int64


In [62]:
healthcare_indicators = [
    "admission_type",
    "admission_status",
    "bed_status",
    "gender",
    "blood_group",
    "ward_type",
    "department_type",
    "disease_category",
    "payment_status",
    "payment_mode"
]

print("HEALTHCARE INDICATORS BEFORE NORMALIZATION ")
for col in healthcare_indicators:
    if col in df.columns:
        print("\n---", col, "---")
        display(
            df[col].value_counts()
        )

HEALTHCARE INDICATORS BEFORE NORMALIZATION 

--- admission_type ---


,count
admission_type,
Elective,26923
Emergency,18077



--- admission_status ---


,count
admission_status,
Discharged,45000



--- bed_status ---


...

,count
ward_type,
General,25379
Private,9007
Semi-Private,6574
Icu,4040



--- department_type ---


,count
department_type,
Clinical,45000



--- disease_category ---


,count
disease_category,
Infectious,11248
Surgical,6711
Respiratory,4564
Cardiac,4525
Pediatric,4436
Neurological,2300
Orthopedic,2288
Trauma,2253
Renal,2251


In [63]:
# Normalization
categorical_columns = [
    "admission_type",
    "admission_status",
    "bed_status",
    "gender",
    "ward_type",
    "department_type",
    "disease_category",
    "payment_status",
    "payment_mode"
]

for col in categorical_columns:

    if col in df.columns:

        df[col] = (
            df[col]
            .astype(str)
            .str.strip()
            .str.title()
        )

In [64]:
if "blood_group" in df.columns:
    df["blood_group"] = (
        df["blood_group"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

In [65]:
print("HEALTHCARE INDICATORS AFTER NORMALIZATION")
for col in healthcare_indicators:
    if col in df.columns:
        print("\n---", col, "---")
        display(
            df[col].value_counts()
        )

HEALTHCARE INDICATORS AFTER NORMALIZATION

--- admission_type ---


,count
admission_type,
Elective,26923
Emergency,18077



--- admission_status ---


,count
admission_status,
Discharged,45000



--- bed_status ---


...

,count
ward_type,
General,25379
Private,9007
Semi-Private,6574
Icu,4040



--- department_type ---


,count
department_type,
Clinical,45000



--- disease_category ---


,count
disease_category,
Infectious,11248
Surgical,6711
Respiratory,4564
Cardiac,4525
Pediatric,4436
Neurological,2300
Orthopedic,2288
Trauma,2253
Renal,2251


In [57]:
# Extracting month and year
df["Admission_Month"] = (
    df["admission_date"]
    .dt.month_name()
)
df["Admission_Year"] = (
    df["admission_date"]
    .dt.year
)

In [66]:
# Normalizing Billing Vlaues
billing_columns = [
    "Total_Billing",
    "Insurance_Covered_Amount",
    "Patient_Payable_Amount"
]
for col in billing_columns:
    if col in df.columns:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        )
        df[col] = df[col].round(2)

In [67]:
# Checking negative values
for col in billing_columns:
    if col in df.columns:
        print(
            col,
            "negative values:",
            (df[col] < 0).sum()
        )

Total_Billing negative values: 0
Insurance_Covered_Amount negative values: 0
Patient_Payable_Amount negative values: 0


In [68]:
df["date_of_birth"] = pd.to_datetime(
    df["date_of_birth"],
    errors="coerce"
)
# Calculate Age
today = pd.Timestamp.today()

df["Age"] = (
    today.year - df["date_of_birth"].dt.year
    - (
        (today.month < df["date_of_birth"].dt.month)
        |
        (
            (today.month == df["date_of_birth"].dt.month)
            &
            (today.day < df["date_of_birth"].dt.day)
        )
    )
)

# Convert to nullable integer
df["Age"] = df["Age"].astype("Int64")

# Display calculated Age
display(
    df[
        ["date_of_birth", "Age"]
    ].head(10)
)

,date_of_birth,Age
0,1954-11-02,71
1,1988-02-25,38
2,1944-03-02,82
3,2017-08-19,8
4,2015-05-06,11
5,1960-01-21,66
6,1952-11-17,73
7,2008-05-12,18
8,1973-08-29,52
9,1960-10-08,65


In [69]:
# Dividing the age Group
df["Age Group"] = pd.cut(
    df["Age"],
    bins=[0, 12, 18, 35, 60, 120],
    labels=[
        "Child",
        "Teenager",
        "Young Adult",
        "Adult",
        "Senior"
    ],
    include_lowest=True
)
display(
    df[["Age", "Age Group"]].head(10)
)

,Age,Age Group
0,71,Senior
1,38,Adult
2,82,Senior
3,8,Child
4,11,Child
5,66,Senior
6,73,Senior
7,18,Teenager
8,52,Adult
9,65,Senior


In [70]:
# Creating Billing Category
df["Billing Category"] = pd.cut(
    df["Total_Billing"],
    bins=[0, 10000, 25000, 50000, float("inf")],
    labels=[
        "Low",
        "Medium",
        "High",
        "Very High"
    ],
    include_lowest=True
)

display(
    df[
        ["Total_Billing", "Billing Category"]
    ].head(10)
)

,Total_Billing,Billing Category
0,68483,Very High
1,70917,Very High
2,28137,High
3,80665,Very High
4,54920,Very High
5,71521,Very High
6,13951,Medium
7,46293,High
8,21830,Medium
9,18606,Medium


In [71]:
print("Total Patients:",
      df["patient_id"].nunique())
print("Total Admissions:",
      df["admission_id"].nunique())
print("Total Billing Amount:",
      round(df["Total_Billing"].sum(), 2))
print("Average Billing Amount:",
      round(df["Total_Billing"].mean(), 2))
print("Total Departments:",
      df["department_id"].nunique())
print("Total Wards:",
      df["ward_id"].nunique())
print("Total Beds:",
      df["bed_id"].nunique())
print("Total Diseases:",
      df["disease_id"].nunique())

Total Patients: 23275
Total Admissions: 45000
Total Billing Amount: 1684246109
Average Billing Amount: 37427.69
Total Departments: 6
Total Wards: 27
Total Beds: 145
Total Diseases: 20


In [78]:
# Check final dataset
print("Final dataset shape:", df.shape)
# Save the cleaned and transformed dataset
file_name = "hospital_cleaned.csv"
df.to_csv(file_name, index=False)
print("Dataset saved successfully as:", file_name)

Final dataset shape: (45000, 39)
Dataset saved successfully as: hospital_cleaned.csv


In [80]:
from google.colab import files

files.download("hospital_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>